[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/10_tag_6_8_skip_connections_direkt_und_resblock_trainiert.ipynb)

# Tag 6-8 - 10 Skip Connections direkt und als Residual Block

Eine Skip Connection wird zuerst direkt als Netz formuliert und danach als Residual Block gekapselt. Beide Varianten werden auf einem Bilddatensatz trainiert.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA verfügbar:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_step(model, xb, yb, loss_fn, optimizer):
    model.train()
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()
    logits = model(xb)
    loss = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item(), accuracy_from_logits(logits.detach(), yb)


def evaluate(model, loader, loss_fn=None):
    model.eval()
    total_loss = 0.0
    correct = 0
    seen = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            if loss_fn is not None:
                total_loss += loss_fn(logits, yb).item() * len(xb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            seen += len(xb)
    avg_loss = total_loss / seen if loss_fn is not None else np.nan
    return avg_loss, correct / seen


def train_epochs(model, train_loader, valid_loader, loss_fn, optimizer, epochs=5):
    history = []
    for epoch in range(epochs):
        losses = []
        accs = []
        for xb, yb in train_loader:
            loss, acc = train_step(model, xb, yb, loss_fn, optimizer)
            losses.append(loss)
            accs.append(acc)
        valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
        row = {
            "epoch": epoch + 1,
            "train_loss": float(np.mean(losses)),
            "train_acc": float(np.mean(accs)),
            "valid_loss": float(valid_loss),
            "valid_acc": float(valid_acc),
        }
        history.append(row)
        print(
            f"Epoch {row['epoch']:02d}: "
            f"train_loss={row['train_loss']:.3f}, train_acc={row['train_acc']:.3f}, "
            f"valid_loss={row['valid_loss']:.3f}, valid_acc={row['valid_acc']:.3f}"
        )
    return history


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Daten


In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
train_dataset = TensorDataset(torch.tensor(X_train).unsqueeze(1), torch.tensor(y_train))
valid_loader = DataLoader(TensorDataset(torch.tensor(X_valid).unsqueeze(1), torch.tensor(y_valid)), batch_size=256)

def make_train_loader():
    # Beide Modelle sehen dieselben Mini-Batches in derselben Reihenfolge.
    generator = torch.Generator().manual_seed(RANDOM_STATE)
    return DataLoader(train_dataset, batch_size=64, shuffle=True, generator=generator)


## Skip Connection direkt im Forward Pass


In [ ]:
class DirectSkipCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.conv1 = nn.Conv2d(16, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(16, 10))

    def forward(self, x):
        x = self.stem(x)
        residual = x
        x = torch.relu(self.conv1(x))
        x = self.conv2(x)
        x = torch.relu(x + residual)
        x = self.pool(x)
        return self.head(x)

# Gleicher Seed wie beim ResidualCNN: gleiche Startgewichte.
torch.manual_seed(RANDOM_STATE)
direct_model = DirectSkipCNN().to(device)
print(direct_model)
print("Parameter:", count_params(direct_model))


## Derselbe Block als wiederverwendbare Klasse

Ein Residual Block lernt eine **Korrektur** `F(x)` zur bereits vorhandenen Repräsentation `x`: `ReLU(x + F(x))`. Wenn die zusätzliche Verarbeitung für ein Bildmerkmal nicht hilft, kann der Block näherungsweise `F(x) = 0` lernen. Dann bleibt der direkte Weg `x` erhalten. Das erleichtert auch den Gradientenfluss beim Training, weil Gradienten über diesen direkten Weg rückwärts laufen können.

Hier ist das `ResidualCNN` bewusst **dasselbe Netz** wie `DirectSkipCNN`: eine Eingangs-Conv, genau zwei Conv-Schichten im Skip-Pfad, ein Pooling und derselbe Klassifikationskopf. Beide Modelle haben daher 4.970 trainierbare Parameter. Der Unterschied ist ausschließlich die Darstellung im Code: einmal ist der Skip-Pfad direkt im `forward` formuliert, einmal steckt er in der wiederverwendbaren Klasse `ResidualBlock`.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return torch.relu(x + self.net(x))

class ResidualCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            ResidualBlock(16),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(16, 10),
        )

    def forward(self, x):
        return self.model(x)

# Gleicher Seed wie beim DirectSkipCNN: gleiche Startgewichte.
torch.manual_seed(RANDOM_STATE)
res_model = ResidualCNN().to(device)
print("Parameter ResidualCNN:", count_params(res_model))


## Beide Darstellungen trainieren

Da Architektur, Startgewichte und Reihenfolge der Mini-Batches gleich sind, sollten beide Varianten dieselbe Lernkurve und dieselben Vorhersagen liefern. Das ist die zentrale Aussage dieses Beispiels: `ResidualBlock` ist eine bessere, wiederverwendbare **Code-Repräsentation** desselben Rechenwegs – keine andere Modellarchitektur.

Für ein tieferes Residualnetz würde man anschließend mehrere `ResidualBlock`s hintereinander setzen. Dann müsste die direkte Variante aber dieselbe Zahl an Skip-Pfaden enthalten, damit der Vergleich fair bleibt.


In [ ]:
models = {"DirectSkipCNN": direct_model, "ResidualCNN": res_model}
histories = {}
loss_fn = nn.CrossEntropyLoss()

for name, model in models.items():
    print(f"\n{name} | Parameter: {count_params(model):,}")
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    histories[name] = train_epochs(model, make_train_loader(), valid_loader, loss_fn, optimizer, epochs=8)
    plt.plot([h["valid_acc"] for h in histories[name]], label=name)

plt.xlabel("Epoch")
plt.ylabel("Valid Accuracy")
plt.legend()
plt.show()

print("\nZusammenfassung nach 8 Epochen:")
for name, history in histories.items():
    print(f"{name:14s}: Valid Accuracy = {history[-1]['valid_acc']:.2%} | Parameter = {count_params(models[name]):,}")


## Beispielausgaben: dieselben Vorhersagen in beiden Darstellungen

Beide Zeilen zeigen dieselben zwölf Validierungsbilder. In jedem Titel steht `wahr` für die echte Ziffer, `vorh.` für die Modellvorhersage und die Prozentzahl für die Sicherheit (höchste Softmax-Wahrscheinlichkeit). Grün bedeutet korrekt, Rot bedeutet falsch. Da die Netze mathematisch identisch trainiert wurden, sollten auch alle Titel übereinstimmen.


In [ ]:
valid_images = torch.tensor(X_valid).unsqueeze(1).to(device)
results = {}

for name, model in models.items():
    model.eval()
    with torch.no_grad():
        logits = model(valid_images).cpu().numpy()
    probabilities = torch.softmax(torch.tensor(logits), dim=1).numpy()
    predictions = probabilities.argmax(axis=1)
    results[name] = {"logits": logits, "probabilities": probabilities, "predictions": predictions}
    print(f"{name:14s}: Valid Accuracy = {(predictions == y_valid).mean():.2%}")

rng = np.random.default_rng(RANDOM_STATE)
sample_indices = rng.choice(len(X_valid), size=12, replace=False)
fig, axes = plt.subplots(4, 6, figsize=(12, 8))

for row, name in enumerate(models):
    predictions = results[name]["predictions"]
    probabilities = results[name]["probabilities"]
    for ax, index in zip(axes[row * 2:(row + 1) * 2].ravel(), sample_indices):
        true_label = y_valid[index]
        predicted_label = predictions[index]
        confidence = probabilities[index, predicted_label]
        color = "forestgreen" if true_label == predicted_label else "crimson"
        ax.imshow(X_valid[index], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"wahr: {true_label} | vorh.: {predicted_label}\n{confidence:.0%}", color=color, fontsize=9)
        ax.axis("off")
    axes[row * 2, 0].set_ylabel(name, fontsize=11, fontweight="bold")

fig.suptitle("Dieselben Validierungsbilder: Vorhersagen beider Code-Repräsentationen", fontsize=14)
plt.tight_layout()
plt.show()


### Direkter Gleichheitscheck

Die Ausgabe vergleicht die Logits vor der Softmax-Funktion. Sie ist strenger als ein Vergleich der vorhergesagten Klasse: Ein Unterschied von `0.0` bedeutet, dass beide Code-Repräsentationen für jedes Validierungsbild exakt denselben Zahlenvektor ausgeben.


In [ ]:
direct_logits = results["DirectSkipCNN"]["logits"]
residual_logits = results["ResidualCNN"]["logits"]
direct_predictions = results["DirectSkipCNN"]["predictions"]
residual_predictions = results["ResidualCNN"]["predictions"]

max_logit_difference = np.abs(direct_logits - residual_logits).max()
different_predictions = (direct_predictions != residual_predictions).sum()
print(f"Maximale Logit-Differenz: {max_logit_difference:.1e}")
print(f"Unterschiedliche Vorhersagen: {different_predictions} von {len(y_valid)}")

if np.allclose(direct_logits, residual_logits):
    print("Ergebnis: Beide Darstellungen berechnen nach dem Training dasselbe Netz.")
else:
    print("Hinweis: Eine kleine Differenz deutet auf unterschiedliche Zufallsabläufe oder Hardware-Numerik hin.")
